# Análise de Padrões em Letras de Música usando TSNE, FastText e Classificação de Idioma

## Resumo
Este projeto tem como objetivo explorar padrões em um dataset de letras de músicas, considerando informações como o ano e features criadas a partir da letra. Utilizando técnicas de vetorização, redução de dimensionalidade e visualização, buscamos identificar comportamentos e agrupamentos nas músicas.

## Resultados
Com TfidfVectorizer e TSNE conseguimos dividir entre inglês e portugues. Porém utilizando a camada de embeading do fasttext a divisão ficou muito mais clara. Existem outras idiomas dentro do dataset e os dois metodos encontraram esses idimos.

Agora olhando somente para as musicas em portugues encontramos um grafico 3D interessante, porém não consegui interpletar o comportamento para alguma coisa significativa

## Tecnologias e Bibliotecas Utilizadas
- Python
- pandas
- scikit-learn (TfidfVectorizer, TSNE)
- fasttext e huggingface_hub (para modelos de identificação de idioma)
- transformers (para classificação de texto)
- altair e plotly (visualização interativa)

# Imports

In [1]:
!pip install 'numpy<2.0'
!pip install fasttext

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 36.8 MB/s eta 0:00:00 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.4
    Uninstalling numpy-2.2.4:
      Successfully uninstalled numpy-2.2.4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-2.13.6-py3-none-any.whl.metadata (9.5 kB)
Using cached pybind11-2.13.6-py3-none-any.whl (243 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp310-cp310-macosx_15_0_arm64.whl size=303230 sha256=524fd43d5679a2fcaa9261bad2fbde91c4a6846c82fcfad6d629f357a2cfca7a
  Stored in directory: /Users/vho/Library/Caches/pip/wheels/0d/a2/00/81db54d3e6a8199b829d58e02cec2ddb20ce3e59fad8d3c92a
Successfully built fasttext


In [2]:
import pandas as pd
import altair as alt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TextClassificationPipeline

# Run TSNE

$R^n → R^2$

In [3]:
# 1. Carregar dados
# You need to load the csv file Files>Upload
df = pd.read_csv("musicas_subsampled.csv")

# 2. Vetorize
vectorizer = TfidfVectorizer(lowercase=True, binary=True)
X = vectorizer.fit_transform(df['letra'].fillna(''))
X_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())

# 3. Apply TSNE
tsne = TSNE(n_components=2, random_state=42)
df['tsne_1'], df['tsne_2'] = tsne.fit_transform(X_df).T

In [4]:
chart = alt.Chart(df[['tsne_1', 'tsne_2']]).mark_point().encode(
    x='tsne_1:Q',
    y='tsne_2:Q',
    tooltip=['tsne_1', 'tsne_2']
).interactive()

chart.show()

alt.Chart(...)

# Add language label using huggingface

In [5]:
model_name = 'qanastek/51-languages-classifier'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
classifier = TextClassificationPipeline(model=model, tokenizer=tokenizer)

def inferir_idioma(texto):
    try:
        resultado = classifier(texto, truncation=True, max_length=512)  # para textos longos, limite o tamanho
        idioma = resultado[0]['label']
        return idioma
    except Exception as e:
        return None

# Aplicar ao seu DataFrame
df['idioma'] = df['letra'].apply(inferir_idioma)

# Verificar resultados
df[['nome_musica', 'artista', 'idioma']]

Device set to use mps:0


,nome_musica,artista,idioma
0,"Nem Ouro, Nem Prata",Ruy Maurity,pt-PT
1,"Bola de Meia, Bola de Gude",Milton Nascimento,pt-PT
2,Have I Told You Lately,Rod Stewart,en-US
3,Little Woman,McFly,en-US
4,Samba Makossa,Charlie Brown Jr.,pt-PT
...,...,...,...
495,Você Me Vira a Cabeça (Me Tira do Sério),Alcione,pt-PT
496,An Affair To Remember,Nat King Cole,en-US
497,Tu T'en Vas,Alain Barrière,fr-FR
498,Diamonds,Sam Smith,en-US


In [6]:
selector = alt.selection_multi(fields=['idioma'], bind='legend')

chart = alt.Chart(df[['tsne_1', 'tsne_2', 'idioma']]).mark_point().encode(
    x='tsne_1:Q',
    y='tsne_2:Q',
    color=alt.condition(selector, 'idioma:N', alt.value('lightgray')),
    tooltip=['tsne_1', 'tsne_2', 'idioma']
).add_selection(
    selector
).interactive()

chart.show()

/var/folders/9j/vn430hgj54s92fj0p3wfvhxh0000gn/T/ipykernel_28807/1777587442.py:1: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use selection_point instead.
  selector = alt.selection_multi(fields=['idioma'], bind='legend')
/var/folders/9j/vn430hgj54s92fj0p3wfvhxh0000gn/T/ipykernel_28807/1777587442.py:8: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use add_params instead.
  ).add_selection(


alt.Chart(...)

# Run FastText pre-trained
---
https://huggingface.co/facebook/fasttext-language-identification

In [7]:
import fasttext
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(repo_id="facebook/fasttext-language-identification", filename="model.bin")
ft_model = fasttext.load_model(model_path)

def fasttext_vector(texto):
    try:
        vector = ft_model.get_word_vector(texto)
        return vector
    except Exception:
        return None

df['vector_ft'] = (
    df['letra'].str.replace('\n', ' ', regex=False)
    .apply(fasttext_vector)
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


model.bin:   0%|          | 0.00/1.18G [00:00<?, ?B/s]

In [8]:
df = df[df['vector_ft'].notna()].copy()
explode_vector = df.loc[df['vector_ft'].notna(), 'vector_ft'].apply(pd.Series)
tsne = TSNE(n_components=2, random_state=42)
df['ft_tsne_1'], df['ft_tsne_2'] = tsne.fit_transform(explode_vector).T

In [9]:
selector = alt.selection_multi(fields=['idioma'], bind='legend')

chart = alt.Chart(df[['ft_tsne_1', 'ft_tsne_2', 'idioma']]).mark_point().encode(
    x='ft_tsne_1:Q',
    y='ft_tsne_2:Q',
    color=alt.condition(selector, 'idioma:N', alt.value('lightgray')),
    tooltip=['ft_tsne_1', 'ft_tsne_2', 'idioma']
).add_selection(
    selector
).interactive()

chart.show()

/var/folders/9j/vn430hgj54s92fj0p3wfvhxh0000gn/T/ipykernel_28807/842894327.py:1: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use selection_point instead.
  selector = alt.selection_multi(fields=['idioma'], bind='legend')
/var/folders/9j/vn430hgj54s92fj0p3wfvhxh0000gn/T/ipykernel_28807/842894327.py:8: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use add_params instead.
  ).add_selection(


alt.Chart(...)

In [10]:
import pandas as pd
from sklearn.manifold import TSNE
import plotly.express as px

def count_most_frequent_word(row):
    words = row.split()
    freq = pd.Series(words).value_counts()
    return freq.max()

# Filter and prepare data
plot_df = df[(df['vector_ft'].notna()) & (df['idioma'] == 'pt-PT')].copy()
explode_vector = pd.concat([
    plot_df['vector_ft'].apply(pd.Series),
    plot_df['letra'].str.len(),
    plot_df[['ano']],
    plot_df['letra'].apply(count_most_frequent_word),
    plot_df['letra'].str.count(r'\n')
], axis=1)
explode_vector.columns = [f'col_{i}' for i in range(explode_vector.shape[1])]

# Run t-SNE with 3 components
tsne = TSNE(n_components=3, random_state=42)
tsne_results = tsne.fit_transform(explode_vector)
plot_df['tsne_1'], plot_df['tsne_2'], plot_df['tsne_3'] = tsne_results[:, 0], tsne_results[:, 1], tsne_results[:, 2]

# Create 3D scatter plot with Plotly
fig = px.scatter_3d(
    plot_df,
    x='tsne_1',
    y='tsne_2',
    z='tsne_3',
    color='idioma',
    hover_data=['tsne_1', 'tsne_2', 'tsne_3']
)

fig.show()


In [11]:
import plotly.express as px

def count_most_frequent_word(row):
    words = row.split()
    freq = pd.Series(words).value_counts()
    return freq.max()

# Filter and prepare data
plot_df = df[(df['vector_ft'].notna()) & (df['idioma'] == 'pt-PT')].copy()
explode_vector = pd.concat([
    plot_df['vector_ft'].apply(pd.Series),
    plot_df['letra'].str.len(),
    plot_df[['ano']],
    plot_df['letra'].apply(count_most_frequent_word),
    plot_df['letra'].str.count(r'\n')
], axis=1)
explode_vector.columns = [f'col_{i}' for i in range(explode_vector.shape[1])]

# Run t-SNE with 3 components
tsne = TSNE(n_components=3, random_state=42)
tsne_results = tsne.fit_transform(explode_vector)
plot_df['tsne_1'], plot_df['tsne_2'], plot_df['tsne_3'] = tsne_results[:, 0], tsne_results[:, 1], tsne_results[:, 2]

# Create 3D scatter plot with Plotly
fig = px.scatter_3d(
    plot_df,
    x='tsne_1',
    y='tsne_2',
    z='tsne_3',
    color='ano',
    hover_data=['tsne_1', 'tsne_2', 'tsne_3', 'ano'],
    size_max=.5
)

fig.show()
